# Práctica: contrato y preprocesado con un modelo real

Este notebook repite, en pequeño, lo que haces en `contracts.py` y `preprocess.py`.

**El caso:** un modelo de scikit-learn entrenado con **datos reales de vinos** (el dataset `wine`, que viene dentro de scikit-learn) que predice la **variedad de uva** (`variedad_A`, `variedad_B` o `variedad_C`) a partir de solo **3 medidas**:

| Feature | Qué es | Rango real en el dataset |
| --- | --- | --- |
| `alcohol` | Grado alcohólico (% vol) | 11.0 a 14.8 |
| `malic_acid` | Ácido málico (g/L) | 0.7 a 5.8 |
| `color_intensity` | Intensidad de color | 1.3 a 13.0 |

**Cómo usarlo:**

1. Ejecuta las celdas **en orden**, de arriba abajo (Shift + Enter).
2. Donde ponga `# TU CÓDIGO AQUÍ`, escribe tu solución.
3. Debajo de cada ejercicio hay una **celda de comprobación**. Si todo está bien, imprime `OK`. Si no, te dice qué falla.
4. Las soluciones están **al final**. Mira solo después de intentarlo.

**Cómo abrirlo** (desde la carpeta `starter`, que ya tiene pydantic y scikit-learn):

```bash
uv run --with jupyter jupyter lab
```

O en VS Code: abre el notebook y elige como kernel el `.venv` del starter.

## Parte 0. El modelo (ya hecho, no lo toques)

Esta celda entrena el modelo. Hace lo mismo que el profesor antes de daros el `.joblib`: entrenar con las features **en un orden concreto** y guardar ese orden en `FEATURE_NAMES`.

In [1]:
from sklearn.datasets import load_wine
from sklearn.ensemble import RandomForestClassifier

# El orden en que se entrena el modelo. A partir de aquí es sagrado.
FEATURE_NAMES = ("alcohol", "malic_acid", "color_intensity")

# Datos reales de 178 vinos
datos = load_wine(as_frame=True).frame
X = datos[list(FEATURE_NAMES)].values          # las 3 columnas, EN ESTE ORDEN
nombres = {0: "variedad_A", 1: "variedad_B", 2: "variedad_C"}
y = datos["target"].map(nombres)               # la etiqueta como texto

modelo = RandomForestClassifier(n_estimators=100, random_state=0)
modelo.fit(X, y)

print("Modelo entrenado con", len(X), "vinos")
print("Orden de las features:", FEATURE_NAMES)
print("Clases que conoce:", list(modelo.classes_))

ImportError: load_wine with as_frame=True requires pandas.

## Parte 1. Ver el problema con tus propios ojos

El modelo no entiende nombres: solo una fila de números. Mira lo que pasa si le das **los mismos números en otro orden**.

In [ ]:
fila_correcta = [[13.2, 1.8, 4.4]]     # alcohol, malic_acid, color_intensity
fila_al_reves = [[4.4, 1.8, 13.2]]     # color_intensity, malic_acid, alcohol

print("Orden correcto  ->", modelo.predict(fila_correcta)[0])
print("Orden al revés  ->", modelo.predict(fila_al_reves)[0])
print()
print("Mismo vino, distinta predicción. Y el modelo NO da ningún error.")

**Pregunta para pensar (respuesta en las soluciones):** ¿por qué el modelo no avisa de que los datos están desordenados?

## Ejercicio 1. El contrato de entrada

Escribe la clase `VinoRequest` con Pydantic:

- Los **3 campos** de `FEATURE_NAMES`, todos `float` y obligatorios.
- **Prohibir campos extra.**
- Rangos: ninguno puede ser negativo. Máximos holgados: `alcohol` 20, `malic_acid` 10, `color_intensity` 20.

Pista: es igual que `WineQualityRequest`, pero con 3 campos.

In [ ]:
from pydantic import BaseModel, ConfigDict, Field, ValidationError


class VinoRequest(BaseModel):
    # TU CÓDIGO AQUÍ
    pass

In [ ]:
# COMPROBACIÓN del ejercicio 1 (no la modifiques)
def debe_fallar(datos, campo):
    try:
        VinoRequest(**datos)
    except ValidationError as error:
        assert campo in str(error), f"El error debería mencionar '{campo}'"
        return
    raise AssertionError(f"Estos datos deberían fallar y han pasado: {datos}")

for campo in FEATURE_NAMES:
    assert campo in VinoRequest.model_fields, f"Falta el campo '{campo}' en VinoRequest"

bueno = {"alcohol": "13.2", "malic_acid": "1.8", "color_intensity": "4.4"}
v = VinoRequest(**bueno)
assert v.alcohol == 13.2, "Los textos del CSV ('13.2') deberían convertirse a número"

debe_fallar({**bueno, "bodega": "Rioja"}, "bodega")           # columna extra
debe_fallar({"alcohol": "13.2", "malic_acid": "1.8"}, "color_intensity")   # falta un campo
debe_fallar({**bueno, "alcohol": "-1"}, "alcohol")            # negativo
debe_fallar({**bueno, "alcohol": "99"}, "alcohol")            # demasiado alto
debe_fallar({**bueno, "malic_acid": "mucho"}, "malic_acid")   # texto
print("OK: el contrato funciona")

## Ejercicio 2. Preprocesado, versión 1: a mano

Escribe `preprocess_v1(request)`, que devuelva **una lista** con los 3 valores **en el orden de `FEATURE_NAMES`**, escribiendo cada campo a mano (`request.alcohol`, ...).

In [ ]:
def preprocess_v1(request):
    # TU CÓDIGO AQUÍ
    pass

In [ ]:
# COMPROBACIÓN del ejercicio 2
# Fíjate: creamos el vino con los campos en OTRO orden. Da igual.
r = VinoRequest(color_intensity=4.4, alcohol=13.2, malic_acid=1.8)
resultado = preprocess_v1(r)
assert resultado is not None, "La función no devuelve nada: ¿falta el return?"
assert resultado == [13.2, 1.8, 4.4], f"Esperaba [13.2, 1.8, 4.4] y salió {resultado}"
print("OK:", resultado)

## Ejercicio 3. Preprocesado, versión 2: recorriendo FEATURE_NAMES

Ahora sin escribir los nombres a mano. Recorre `FEATURE_NAMES` con un bucle `for` y usa `getattr(request, name)` para sacar cada valor.

Recuerda: `getattr(request, "alcohol")` es lo mismo que `request.alcohol`.

In [ ]:
def preprocess_v2(request):
    values = []
    # TU CÓDIGO AQUÍ (un for y un append)
    return values

In [ ]:
# COMPROBACIÓN del ejercicio 3
r = VinoRequest(color_intensity=4.4, alcohol=13.2, malic_acid=1.8)
resultado = preprocess_v2(r)
assert resultado == [13.2, 1.8, 4.4], f"Esperaba [13.2, 1.8, 4.4] y salió {resultado}"
assert resultado == preprocess_v1(r), "v1 y v2 deberían dar exactamente lo mismo"
print("OK:", resultado)

## Ejercicio 4. La cajita con `as_vector()`

En la práctica, el test llama a `features.as_vector()`. Escribe:

1. La clase `VinoFeatures` con `@dataclass`, un campo `values: list[float]` y un método `as_vector()` que devuelva la lista.
2. La función `preprocess_vino_request(request)`, que construya la lista (como en la v2) y devuelva un `VinoFeatures`.

In [ ]:
from dataclasses import dataclass


# TU CÓDIGO AQUÍ: la clase VinoFeatures


def preprocess_vino_request(request):
    # TU CÓDIGO AQUÍ
    pass

In [ ]:
# COMPROBACIÓN del ejercicio 4 (igual que el test de la práctica)
r = VinoRequest(alcohol=13.2, malic_acid=1.8, color_intensity=4.4)
features = preprocess_vino_request(r)
assert features is not None, "La función no devuelve nada: ¿falta el return?"
assert hasattr(features, "as_vector"), "Lo que devuelves no tiene el método as_vector()"
assert features.as_vector() == [13.2, 1.8, 4.4], f"Salió {features.as_vector()}"
print("OK:", features)

## Ejercicio 5. Todo junto: de los datos a la predicción

Escribe `predecir(datos)`, que reciba un diccionario como los de una fila de CSV y:

1. Lo valide con `VinoRequest`.
2. Lo preprocese con `preprocess_vino_request`.
3. Le pida la predicción al modelo con `modelo.predict(...)`.
4. Devuelva **solo el texto** de la variedad (por ejemplo, `"variedad_A"`).

Cuidado: el modelo espera una **lista de filas**, así que la fila va entre corchetes extra: `[features.as_vector()]`. Y `predict` devuelve una lista: coge el elemento `[0]`.

In [ ]:
def predecir(datos):
    # TU CÓDIGO AQUÍ (4 líneas)
    pass

In [ ]:
# COMPROBACIÓN del ejercicio 5
fila = {"alcohol": "13.2", "malic_acid": "1.8", "color_intensity": "4.4"}
resultado = predecir(fila)
assert resultado == "variedad_A", f"Esperaba 'variedad_A' y salió {resultado!r}"

# Aunque el diccionario venga en otro orden, la predicción es la correcta
fila_desordenada = {"color_intensity": "4.4", "alcohol": "13.2", "malic_acid": "1.8"}
assert predecir(fila_desordenada) == "variedad_A", "El orden del diccionario no debería importar"
print("OK: la predicción es", resultado)

## Ejercicio 6 (bonus). Varias filas, todo o nada

Esto es la idea de `predict_file.py`. Escribe `predecir_lote(filas)`, que reciba una **lista** de diccionarios con una clave extra `sample_id` y:

- Aparte `sample_id` de cada fila con `fila.pop("sample_id")`, porque no es una feature.
- Prediga cada fila y guarde en una lista `{"sample_id": ..., "variedad": ...}`.
- Si **cualquier** fila falla, lance el error **sin devolver nada parcial**.

Trabaja con una **copia** de cada fila (`dict(fila)`), para no modificar los datos originales con el `pop`.

In [ ]:
def predecir_lote(filas):
    resultados = []
    # TU CÓDIGO AQUÍ
    return resultados

In [ ]:
# COMPROBACIÓN del ejercicio 6
lote_bueno = [
    {"sample_id": "v-001", "alcohol": "13.2", "malic_acid": "1.8", "color_intensity": "4.4"},
    {"sample_id": "v-002", "alcohol": "12.4", "malic_acid": "1.1", "color_intensity": "2.9"},
    {"sample_id": "v-003", "alcohol": "13.0", "malic_acid": "3.9", "color_intensity": "9.0"},
]
salida = predecir_lote(lote_bueno)
assert [s["sample_id"] for s in salida] == ["v-001", "v-002", "v-003"], f"Salió {salida}"
assert [s["variedad"] for s in salida] == ["variedad_A", "variedad_B", "variedad_C"], f"Salió {salida}"
assert "sample_id" in lote_bueno[0], "Has modificado la fila original: usa dict(fila) para copiarla"

lote_malo = [lote_bueno[0], {**lote_bueno[1], "alcohol": "invalid"}]
try:
    predecir_lote(lote_malo)
    raise AssertionError("Con una fila inválida debería lanzar un error")
except ValidationError:
    pass
print("OK:", salida)

## Reto final (sin solución): cambia el ejemplo

Para comprobar que dominas el método, repite los ejercicios 1 a 5 **con otras features**. Cambia `FEATURE_NAMES` en la Parte 0 por:

```python
FEATURE_NAMES = ("flavanoids", "proline", "hue")
```

Rangos reales: `flavanoids` 0.3 a 5.1, `proline` 278 a 1680, `hue` 0.5 a 1.7. Ten en cuenta que las celdas de comprobación usan las features antiguas, así que tendrás que comprobarlo tú: valida un vino, preprocesa y predice.

---
# Soluciones

Mira cada una solo después de haberlo intentado.

**Parte 1. ¿Por qué el modelo no avisa?** Porque recibe 3 números y no sabe qué significa cada uno. Para él, `[4.4, 1.8, 13.2]` es una fila perfectamente normal de 3 números: lee el 4.4 como alcohol y el 13.2 como intensidad de color. Por eso el orden lo tiene que garantizar **nuestro código** (`FEATURE_NAMES` + preprocesado), no el modelo.

**Solución del ejercicio 1**

In [ ]:
# Solución (puedes ejecutarla para comprobar)
from pydantic import BaseModel, ConfigDict, Field, ValidationError


class VinoRequest(BaseModel):
    model_config = ConfigDict(extra="forbid")

    alcohol: float = Field(ge=0, le=20)
    malic_acid: float = Field(ge=0, le=10)
    color_intensity: float = Field(ge=0, le=20)

**Solución del ejercicio 2**

In [ ]:
# Solución (puedes ejecutarla para comprobar)
def preprocess_v1(request):
    return [request.alcohol, request.malic_acid, request.color_intensity]

**Solución del ejercicio 3**

In [ ]:
# Solución (puedes ejecutarla para comprobar)
def preprocess_v2(request):
    values = []
    for name in FEATURE_NAMES:
        values.append(getattr(request, name))
    return values

**Solución del ejercicio 4**

In [ ]:
# Solución (puedes ejecutarla para comprobar)
from dataclasses import dataclass


@dataclass
class VinoFeatures:
    values: list[float]

    def as_vector(self) -> list[float]:
        return self.values


def preprocess_vino_request(request):
    values = [getattr(request, name) for name in FEATURE_NAMES]
    return VinoFeatures(values=values)

**Solución del ejercicio 5**

In [ ]:
# Solución (puedes ejecutarla para comprobar)
def predecir(datos):
    request = VinoRequest(**datos)
    features = preprocess_vino_request(request)
    prediccion = modelo.predict([features.as_vector()])
    return str(prediccion[0])

**Solución del ejercicio 6**

In [ ]:
# Solución (puedes ejecutarla para comprobar)
def predecir_lote(filas):
    resultados = []
    for fila in filas:
        fila = dict(fila)                    # copia
        sample_id = fila.pop("sample_id")    # apartar el identificador
        variedad = predecir(fila)            # si falla, el error sube y no se devuelve nada
        resultados.append({"sample_id": sample_id, "variedad": variedad})
    return resultados